# Run Full Benchmark Suite

This notebook runs multiple embedding and model combinations to build a comprehensive benchmark.

## Setup

In [ ]:
# Uncomment for Google Colab
# !pip install -q gensim transformers torch scikit-learn

# Mount Drive or download dataset (see 01_baseline_tfidf_logreg.ipynb for details)
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'src'))

from imdb_benchmark.data_loader import load_imdb_data
from imdb_benchmark.embeddings.registry import create_embedding, list_embeddings
from imdb_benchmark.models.registry import create_model, list_models
from imdb_benchmark.tuning import CVEvaluator, aggregate_cv_results

import pandas as pd
import logging

logging.basicConfig(level=logging.INFO)

## Configure Experiment

In [ ]:
# Update this path!
DATA_PATH = "/content/aclImdb"  # For Colab
# DATA_PATH = "/path/to/aclImdb"  # For local

# Define experiments to run
EXPERIMENTS = [
    # TF-IDF experiments
    {"embedding": "tfidf_unigram", "model": "logreg"},
    {"embedding": "tfidf_bigram", "model": "logreg"},
    {"embedding": "tfidf_bigram", "model": "rf"},
    {"embedding": "tfidf_bigram", "model": "adaboost"},
    
    # Word2Vec experiments
    {"embedding": "w2v_cbow", "model": "logreg"},
    {"embedding": "w2v_skipgram", "model": "logreg"},
    {"embedding": "w2v_cbow", "model": "rf"},
    
    # BERT experiments (these will take longer!)
    # {"embedding": "bert_base", "model": "logreg"},
    # {"embedding": "distilbert_base", "model": "logreg"},
    
    # LSTM experiments (end-to-end)
    # {"embedding": "lstm", "model": "lstm"},  # Note: LSTM doesn't need separate embedding
]

# CV configuration
N_FOLDS = 5
SEEDS = [42, 123, 456, 789]

## List Available Options

In [ ]:
print("Available Embeddings:")
embeddings = list_embeddings()
for emb_type, emb_list in embeddings.items():
    print(f"  {emb_type}: {emb_list}")

print("\nAvailable Models:")
models = list_models()
for model_type, model_list in models.items():
    print(f"  {model_type}: {model_list}")

## Load Data

In [ ]:
X_train, y_train, X_test, y_test = load_imdb_data(DATA_PATH)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## Run Experiments

In [ ]:
all_results = []

for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*80}")
    print(f"Experiment {i}/{len(EXPERIMENTS)}: {exp['embedding']} + {exp['model']}")
    print(f"{'='*80}\n")
    
    # Create factories
    def embedding_fn():
        return create_embedding(exp['embedding'])
    
    def model_fn():
        return create_model(exp['model'], random_state=SEEDS[0])
    
    # Create evaluator
    evaluator = CVEvaluator(
        n_folds=N_FOLDS,
        seeds=SEEDS,
        tune_hyperparams=False,
    )
    
    # Run CV
    try:
        results_df = evaluator.evaluate(
            embedding_fn=embedding_fn,
            model_fn=model_fn,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            embedding_name=exp['embedding'],
            model_name=exp['model'],
        )
        all_results.append(results_df)
        
        # Print summary
        print(f"\nResults: Acc={results_df['accuracy'].mean():.4f}, F1={results_df['f1'].mean():.4f}")
    
    except Exception as e:
        print(f"ERROR: Experiment failed: {e}")
        import traceback
        traceback.print_exc()

# Combine all results
if all_results:
    combined_results = pd.concat(all_results, ignore_index=True)
    print(f"\nTotal results: {len(combined_results)} evaluations")
else:
    print("\nNo results collected!")
    combined_results = pd.DataFrame()

## View Summary

In [ ]:
if not combined_results.empty:
    summary = aggregate_cv_results(combined_results)
    print("\nBenchmark Summary (sorted by F1 score):")
    print(summary[['embedding', 'model', 'accuracy_mean', 'f1_mean']].to_string(index=False))
else:
    print("No results to summarize.")

## Save Results

In [ ]:
if not combined_results.empty:
    # Create output directory
    output_dir = Path("reports/results")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save detailed results
    results_file = output_dir / "results_long.csv"
    combined_results.to_csv(results_file, index=False)
    print(f"Results saved to {results_file}")
    
    # Save summary
    summary_file = output_dir / "results_summary.csv"
    summary.to_csv(summary_file, index=False)
    print(f"Summary saved to {summary_file}")
    
    # Generate markdown table
    !python scripts/build_results_table.py
else:
    print("No results to save.")

## Export to Google Drive (Optional)

In [ ]:
# Copy results to Google Drive for persistence
# !cp -r reports/results /content/drive/MyDrive/imdb_benchmark_results/

## Download Results (for local commit)

In [ ]:
# Download results files to commit locally
# from google.colab import files
# files.download('reports/results/results_long.csv')
# files.download('reports/results/results_summary.csv')
# files.download('reports/results/results_table.md')